In [1]:
import os

data_path = r"C:\Users\Lenovo\OneDrive\Desktop\dataset\RAVDESS"

files = []
labels = []

for root, dirs, filenames in os.walk(data_path):
    for file in filenames:
        if file.endswith(".wav"):
            parts = file.split("-")
            emotion = int(parts[2])
            
            if emotion in [5, 6, 7]:
                label = 1   # scam
            else:
                label = 0   # not scam
            
            files.append(os.path.join(root, file))
            labels.append(label)

print(len(files), "files loaded")
print("Sample file:", files[0])
print("Sample label:", labels[0])

2880 files loaded
Sample file: C:\Users\Lenovo\OneDrive\Desktop\dataset\RAVDESS\Actor_01\03-01-01-01-01-01-01.wav
Sample label: 0


In [4]:
import librosa
import numpy as np

def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=None)
        
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs.T, axis=0)
        
        return mfccs_mean
    except:
        return None

X = []
y = []

for file, label in zip(files, labels):
    features = extract_features(file)
    if features is not None:
        X.append(features)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)
print("Labels shape:", y.shape)

Feature shape: (2880, 13)
Labels shape: (2880,)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# model
model = LogisticRegression(max_iter=1000)

# train
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)

# results
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7204861111111112
              precision    recall  f1-score   support

           0       0.75      0.82      0.78       358
           1       0.65      0.56      0.60       218

    accuracy                           0.72       576
   macro avg       0.70      0.69      0.69       576
weighted avg       0.72      0.72      0.72       576



In [6]:
sample = X[0].reshape(1, -1)
prediction = model.predict(sample)

print("Prediction:", "Scam" if prediction[0] == 1 else "Not Scam")

Prediction: Not Scam
